# Adversarial Temporal TTA Benchmark (Qwen2.5-7B)

Uses a quantized 7B model + TTA retrieval layer to test whether splitting temporal reasoning
(retrieval handled by TTA systems A-D, generation handled by the LLM) can beat a baseline.

**Core idea:** The TTA layer handles temporal reasoning. The model just generates text.
This splits the cognitive load: the model doesn't need perfect temporal memory,
it just needs to follow instructions from the TTA-retrieved context.

## Cell 1: Setup & Install Dependencies

In [ ]:
# Install dependencies
!pip install -q pandas matplotlib transformers bitsandbytes accelerate scipy tqdm scikit-learn sentence-transformers

import os
os.environ["HF_TOKEN"] = "***REDACTED***"

print("Setup complete.")

## Cell 2: Clone Repo & Generate Adversarial Data

In [ ]:
%cd /content
!git clone https://github.com/toxzak-svg/time.git
%cd time
!python scripts/generate_adversarial_temporal.py --out-dir benchmarks
print("Generated adversarial facts and questions")

## Cell 3: Load Model & Tokenizer

Uses Q4_K_M quantization (bitsandbytes 4-bit NF4) -- fits in ~4 GB VRAM on a T4.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Model priority list -- Qwen first, fallbacks if unavailable
MODEL_CANDIDATES = [
    "Qwen/Qwen2.5-7B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "google/gemma-2-2b-it",
    "meta-llama/Llama-3.1-8B-Instruct",
]

model_id = None
for candidate in MODEL_CANDIDATES:
    try:
        tokenizer = AutoTokenizer.from_pretrained(candidate, token=os.environ["HF_TOKEN"])
        model_id = candidate
        print(f"Successfully loaded tokenizer for: {candidate}")
        break
    except Exception as e:
        print(f"Could not load tokenizer for {candidate}: {e}")

if model_id is None:
    raise RuntimeError("Could not load any model from the candidate list.")

# Q4_K_M quantization via bitsandbytes
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    token=os.environ["HF_TOKEN"],
)
model.eval()

param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"Model loaded: {model_id}")
print(f"Parameter memory: {param_bytes / 1e9:.2f} GB")

## Cell 4: TTA Retrieval Systems (A / B / C / D / D_revised)

Pure-Python temporal retrievers copied from `scripts/run_benchmark.py`.

In [ ]:
from typing import Optional

# ---- Utility functions ----

def is_valid_as_of(fact: dict, day: int) -> bool:
    """Check whether a fact is valid as of the given day."""
    return (
        fact["t_valid_from"] <= day
        and (fact["t_valid_until"] is None or day <= fact["t_valid_until"])
    )

def get_subject(content: str) -> str:
    """Extract subject from fact content. Format: 'domain:subject:...'"""
    parts = content.split(":")
    return parts[1] if len(parts) >= 2 else content


# ---- System A: Naive latest-match (ignores time) ----

def retrieve_plain(facts: list[dict], domain: str, subject: str, as_of_day: int, **kwargs) -> Optional[str]:
    """Baseline A: naive latest match, ignores time entirely."""
    cands = [f for f in facts if f["domain"] == domain and get_subject(f["content"]) == subject]
    if not cands:
        return None
    return max(cands, key=lambda x: x["t_valid_from"])["content"]


# ---- System B: Temporal reranking (recency + validity, no decay) ----

def retrieve_temporal_rerank(facts: list[dict], domain: str, subject: str, as_of_day: int, **kwargs) -> Optional[str]:
    """Baseline B: temporal-aware retrieval with recency + validity scoring."""
    cands = [f for f in facts if f["domain"] == domain and get_subject(f["content"]) == subject]
    if not cands:
        return None

    def score(f: dict) -> float:
        if not is_valid_as_of(f, as_of_day):
            return -1.0
        recency = 1.0 / (1.0 + (as_of_day - f["t_valid_from"]) * 0.1)
        return f.get("confidence", 1.0) * recency

    ranked = sorted(cands, key=score, reverse=True)
    return ranked[0]["content"] if ranked and score(ranked[0]) >= 0 else None


# ---- System C: Time constraint only ----

def retrieve_time_constraint(facts: list[dict], domain: str, subject: str, as_of_day: int, **kwargs) -> Optional[str]:
    """Baseline C: explicit time filter only -- picks the most recent fact valid at as_of_day."""
    cands = [
        f for f in facts
        if f["domain"] == domain
        and get_subject(f["content"]) == subject
        and is_valid_as_of(f, as_of_day)
    ]
    if not cands:
        return None
    return max(cands, key=lambda x: x["t_valid_from"])["content"]


# ---- System D: Full TTA (truth intervals + domain-adaptive decay) ----

def retrieve_tta(facts: list[dict], domain: str, subject: str, as_of_day: int, **kwargs) -> Optional[str]:
    """System D: full TTA with truth intervals and domain-adaptive decay."""
    cands = [f for f in facts if f["domain"] == domain and get_subject(f["content"]) == subject]
    if not cands:
        return None

    half_life = {"slow": 45, "medium": 20, "fast": 7}

    def score(f: dict) -> float:
        if not is_valid_as_of(f, as_of_day):
            return -1.0
        age = max(0, as_of_day - f["t_valid_from"])
        decay = 0.5 ** (age / half_life.get(f["domain"], 20))
        return f.get("confidence", 1.0) * decay

    ranked = sorted(cands, key=score, reverse=True)
    return ranked[0]["content"] if ranked and score(ranked[0]) >= 0 else None


# ---- System D_revised: Validity-only + confidence tiebreak (no decay) ----

def retrieve_tta_improved(facts: list[dict], domain: str, subject: str, as_of_day: int, **kwargs) -> Optional[str]:
    """System D_revised: time constraint + confidence tiebreaker (no decay function)."""
    cands = [f for f in facts if f["domain"] == domain and get_subject(f["content"]) == subject]
    if not cands:
        return None

    valid_cands = [f for f in cands if is_valid_as_of(f, as_of_day)]
    if not valid_cands:
        return None

    return max(valid_cands, key=lambda x: (x["t_valid_from"], x.get("confidence", 1.0)))["content"]


# ---- Map system names to resolvers ----
SYSTEMS = {
    "A": retrieve_plain,
    "B": retrieve_temporal_rerank,
    "C": retrieve_time_constraint,
    "D": retrieve_tta,
    "D_revised": retrieve_tta_improved,
}

print("TTA retrieval systems loaded: A, B, C, D, D_revised")

## Cell 5: Load Adversarial Benchmark Data

In [ ]:
import json
from pathlib import Path

BENCH_DIR = Path("benchmarks")

def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


facts_path = BENCH_DIR / "adversarial_temporal_facts.jsonl"
questions_path = BENCH_DIR / "adversarial_temporal_questions.jsonl"

facts = read_jsonl(facts_path)
questions = read_jsonl(questions_path)

print(f"Loaded {len(facts)} facts, {len(questions)} questions")

# Preview
print("\nSample fact:", facts[0])
print("\nSample question:", questions[0])

## Cell 6: LLM-Powered Answer Generation

Given the TTA-retrieved fact content, use the LLM to produce the final answer.
The prompt is minimal -- the TTA layer already did the temporal reasoning.

In [ ]:
def build_prompt(domain: str, subject: str, as_of_day: int, fact_content: str) -> str:
    """Build the prompt sent to the LLM given a single retrieved fact."""
    return (
        f"You are answering a temporal question about {subject} in the {domain} domain.\n\n"
        f"Retrieved fact: {fact_content}\n\n"
        f"Question: What was the value or state of {subject} as of day {as_of_day}?\n\n"
        f"Respond with ONLY the exact value or state from the retrieved fact. "
        f'If the fact does not apply to day {as_of_day}, respond with "unknown".'
        f"Do not add any explanation. Answer:"
    )

def generate_with_context(fact_content: str, domain: str, subject: str, as_of_day: int) -> str:
    """Use the LLM to generate the answer given one retrieved fact."""
    if fact_content is None:
        return "unknown"
    
    prompt = build_prompt(domain, subject, as_of_day, fact_content)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("LLM generation helper ready.")

## Cell 7: Run Benchmark -- LLM + TTA Systems

For each system, measure:
1. **Retrieval time** -- pure Python, no GPU
2. **Generation time** -- LLM forward pass
3. **Accuracy** -- does the LLM's final answer match the ground truth?

In [ ]:
import time
from tqdm.auto import tqdm


def evaluate_system_llm(
    system_name: str,
    resolver,
    facts: list[dict],
    questions: list[dict],
) -> dict:
    """Evaluate one TTA system + LLM generation on all questions."""
    correct = 0
    retrieval_times = []
    generation_times = []
    total = len(questions)

    for q in tqdm(questions, desc=system_name):
        domain = q["domain"]
        subject = q["subject"]
        as_of_day = q["as_of_day"]
        ground_truth = q.get("answer")

        # --- TTA retrieval (pure Python) ---
        t0 = time.time()
        retrieved = resolver(facts, domain, subject, as_of_day)
        retrieval_time = time.time() - t0
        retrieval_times.append(retrieval_time)
        
        # --- LLM generation ---
        t1 = time.time()
        if retrieved:
            answer = generate_with_context(retrieved, domain, subject, as_of_day)
        else:
            answer = "unknown"
        generation_time = time.time() - t1
        generation_times.append(generation_time)
        
        # --- Accuracy check ---
        pred = answer.strip().lower()
        gt = str(ground_truth).strip().lower() if ground_truth is not None else ""
        if pred == gt:
            correct += 1
    
    avg_retrieval = sum(retrieval_times) / len(retrieval_times) if retrieval_times else 0
    avg_generation = sum(generation_times) / len(generation_times) if generation_times else 0
    avg_total = avg_retrieval + avg_generation

    return {
        "system": system_name,
        "accuracy": round(correct / total, 4),
        "avg_retrieval_s": round(avg_retrieval, 4),
        "avg_generation_s": round(avg_generation, 4),
        "avg_total_s": round(avg_total, 4),
        "correct": correct,
        "total": total,
    }


# Run all systems
results = []
for sys_name, resolver in SYSTEMS.items():
    result = evaluate_system_llm(sys_name, resolver, facts, questions)
    results.append(result)
    print(
        f"  {sys_name}: acc={result['accuracy']:.4f} | "
        f"retrieval={result['avg_retrieval_s']:.4f}s | "
        f"generation={result['avg_generation_s']:.4f}s | "
        f"total={result['avg_total_s']:.4f}s"
    )

print("\nAll LLM+TTA systems done.")

## Cell 8: Pure Retrieval Baseline (no LLM)

Run the TTA systems without LLM generation to isolate the LLM's contribution.
Here the retrieved fact content is compared directly against the ground-truth content string.

In [ ]:
def evaluate_retrieval_only(
    system_name: str,
    resolver,
    facts: list[dict],
    questions: list[dict],
) -> dict:
    """Evaluate retrieval only -- no LLM generation."""
    correct = 0
    total = len(questions)

    for q in questions:
        pred = resolver(facts, q["domain"], q["subject"], q["as_of_day"])
        gt = q.get("answer")
        if gt is None:
            correct += 1 if pred is None else 0
        elif pred == gt:
            correct += 1

    return {
        "system": system_name,
        "accuracy": round(correct / total, 4),
        "correct": correct,
        "total": total,
    }


baseline_results = []
for sys_name, resolver in SYSTEMS.items():
    result = evaluate_retrieval_only(sys_name, resolver, facts, questions)
    baseline_results.append(result)
    print(f"  {sys_name}: retrieval-only acc={result['accuracy']:.4f}")

print("\nPure retrieval baseline done.")

## Cell 9: Save Results & Plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Combine LLM+TTA results with retrieval-only baseline
df_llm = pd.DataFrame(results)
df_baseline = pd.DataFrame(baseline_results)

# Merge for comparison
df = df_llm.merge(df_baseline, on="system", suffixes=("_llm", "_baseline"))

# Save CSV
csv_path = RESULTS_DIR / "adversarial_tta_7b_results.csv"
df.to_csv(csv_path, index=False)
print(f"Saved results to {csv_path}")

# ---- Plot ----
colors = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#e67e22"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Accuracy comparison
ax1 = axes[0]
x = range(len(df))
width = 0.35
ax1.bar([i - width/2 for i in x], df["accuracy_baseline"], width, label="Retrieval-only", color="#95a5a6")
ax1.bar([i + width/2 for i in x], df["accuracy_llm"], width, label="LLM + TTA", color="#2c3e50")
ax1.set_xticks(x)
ax1.set_xticklabels(df["system"])
ax1.set_ylabel("Accuracy")
ax1.set_title("Accuracy: Retrieval-only vs LLM+TTA")
ax1.legend()
ax1.set_ylim(0, 1)

# 2. Timing breakdown
ax2 = axes[1]
ax2.bar(x, df["avg_retrieval_s"], width=0.6, label="Retrieval", color="#3498db")
ax2.bar(x, df["avg_generation_s"], width=0.6, bottom=df["avg_retrieval_s"], label="Generation", color="#e67e22")
ax2.set_xticks(x)
ax2.set_xticklabels(df["system"])
ax2.set_ylabel("Seconds")
ax2.set_title("Avg Time per Query")
ax2.legend()

# 3. Quality vs Speed tradeoff
ax3 = axes[2]
ax3.scatter(df["avg_total_s"], df["accuracy_llm"], c=colors[:len(df)], s=100, zorder=3)
for i, row in df.iterrows():
    ax3.annotate(row["system"], (row["avg_total_s"], row["accuracy_llm"]),
                 fontsize=9, xytext=(5, 5), textcoords="offset points")
ax3.set_xlabel("Avg Total Time (s)")
ax3.set_ylabel("Accuracy")
ax3.set_title("Quality vs Speed Tradeoff")
ax3.grid(True, alpha=0.3)

plt.tight_layout()

fig_path = RESULTS_DIR / "adversarial_tta_7b.png"
plt.savefig(fig_path, dpi=150)
print(f"Saved figure to {fig_path}")
plt.show()

print("\nFinal results table:")
print(df[["system", "accuracy_baseline", "accuracy_llm", "avg_retrieval_s", "avg_generation_s", "avg_total_s"]].to_string(index=False))